Here, I do a similar analysis but compare the output before/after the fix.

In [1]:
library(tidyverse)
library(data.table)
library(cowplot)
library(arrow)
library(Rsamtools)
library(GenomicRanges)
library(GenomicAlignments)
library(RColorBrewer)

base_bam_dir <- "../exp/bam/mm2_cutadapt"
fig_dir <- file.path("../exp/fig/clipping/cutadapt")
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

depth_dir <- file.path(base_bam_dir, "all/depth/")
depth_ar_ds <- file.path(base_bam_dir, "all/depth/arrow/")
clip_end_dir <- file.path(base_bam_dir, "clipped", "clipped_ends")
output_dataset_dir <- "../tmp/clipping_analysis/cutadapt"
dir.create(depth_ar_ds, showWarnings = FALSE, recursive = TRUE)
dir.create(output_dataset_dir, showWarnings = FALSE, recursive = TRUE)

is_position_csv <- '../data/IS_positions.csv'
is_pos_mds_csv <- '../data/IS_positions_in_ref_genome_MDS.csv'

output_summary_csv <- file.path("../exp/dat/", "clipping_summary_cutadapt.csv")
dir.create(dirname(output_summary_csv), showWarnings = FALSE, recursive = TRUE)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.3     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.3     ✔ tibble    3.2.1
✔ lubridate 1.9.2     ✔ tidyr     1.3.0
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose



Attaching package: ‘cowplot’


The following object is masked from ‘package:lubridate’:

    stamp



Attaching package: ‘arrow’


The follow

In [2]:
ids = read_csv("../data/File_list_20250204.csv") %>% 
	mutate(sample = IS_Detect_ID, Sample_2 = sample_name_raw) %>%
	distinct(sample, .keep_all = TRUE) %>%
	filter(gen != "Anc")
ids %>% head

Rows: 133 Columns: 15
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (8): IS_Detect_ID, gen, file_name, Anc, Prefix, sample_name_raw, Contig_...
dbl (5): ParentLine, SubLine, RecA, Folder, Folder_check
lgl (2): Complete, folder_err

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


IS_Detect_ID,ParentLine,SubLine,gen,file_name,Anc,RecA,Prefix,sample_name_raw,Contig_Date,Complete,Folder,Folder_check,folder_err,File,sample,Sample_2
<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<lgl>,<dbl>,<dbl>,<lgl>,<chr>,<chr>,<chr>
L01_Anc,1,1,FACS,20231004/L01_Anc_m1.fasta,R01,1,L01-1,L01_Anc,20220418,TRUE,20231004,20231004,NA,L01_Anc_m1.fasta,L01_Anc,L01_Anc
L02_Anc,2,1,FACS,20231003/L02_Anc.fasta,R02,1,L02-1,L02_Anc,20220418,TRUE,20231003,20231003,NA,L02_Anc.fasta,L02_Anc,L02_Anc
L03_Anc,3,1,FACS,20240703/L03_Anc_m1.fasta,R03,1,L03-1,L03_Anc,20220418,TRUE,20240703,20240703,NA,L03_Anc_m1.fasta,L03_Anc,L03_Anc
L04_Anc,4,1,FACS,20231004/L04_Anc_m1.fasta,R04,1,L04-1,L04_Anc,20230717,TRUE,20231004,20231004,NA,L04_Anc_m1.fasta,L04_Anc,L04_Anc
L05_Anc,5,1,FACS,20231004/L05_Anc_m1.fasta,R05,1,L05-1,L05_Anc,20230909,TRUE,20231004,20231004,NA,L05_Anc_m1.fasta,L05_Anc,L05_Anc
L06_Anc,6,1,FACS,20231004/L06_Anc_m1.fasta,R06,1,L06-1,L06_Anc,20230717,TRUE,20231004,20231004,NA,L06_Anc_m1.fasta,L06_Anc,L06_Anc


In [3]:
#if (!dir.exists(output_dataset_dir)) {
if (TRUE) {
for (sample in ids$sample) {
  # read depth
  df_ <- read_tsv(file.path(depth_dir, paste0(sample, "_depth.txt")), show_col_types = FALSE)
  colnames(df_) <- c("chr", "pos", "depth")
  # save to arrow dataset
  write_feather(df_, file.path(depth_ar_ds, paste0(sample, "_depth.feather")), compression = "zstd")
}
rm(df_)
}

In [4]:
ds <- open_dataset(file.path(depth_ar_ds), format="arrow")
ds %>% head %>% collect

,chr,pos,depth
,<chr>,<dbl>,<dbl>
1,L01-1_G08,2,23
2,L01-1_G08,3,24
3,L01-1_G08,4,26
4,L01-1_G08,5,27
5,L01-1_G08,6,27
6,L01-1_G08,7,27


In [5]:
ds %>% filter(chr == "L11-4_G08") %>% glimpse

FileSystemDataset with 99 Feather files (query)
4,007,861 rows x 3 columns
$ chr   <string> "L11-4_G08", "L11-4_G08", "L11-4_G08", "L11-4_G08", "L11-4_G08"…
$ pos   <double> 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,…
$ depth <double> 19, 19, 19, 19, 22, 22, 22, 22, 22, 22, 22, 22, 21, 21, 22, 22,…
Call `print()` for query details


In [6]:
window_size <- 1000

#if (!dir.exists(output_dataset_dir)) {
dir.create(file.path(output_dataset_dir), showWarnings = FALSE)
#for (i in 1:length(ids$sample)) {
if (TRUE){
for (i in 1:length(ids$sample)) {
  clip_file <- file.path(clip_end_dir, paste0("filtered_clipped_ends_", ids$sample[i], ".txt"))
  # raise error and continue
  if (!file.exists(clip_file)) {
    print(paste0("File not found: ", clip_file))
    next
  }
  clipping_ends <- fread(clip_file)
  #read_depth <- filter(read_depth_master, chr == ids$sample[i])
  read_depth <- ds %>% filter(chr == ids$sample[i]) %>% collect

  # Assign bins
  clipping_ends$Bin <- floor(clipping_ends$Position / window_size) * window_size
  read_depth$Bin <- floor(read_depth$pos / window_size) * window_size

  # Count clipping events per bin
  clipping_counts <- clipping_ends %>%
	group_by(Chromosome, Bin, ReadName) %>%
  summarise(raw_count = n(), .groups = "drop") %>%
  mutate(count = ifelse(raw_count > 2, 2, raw_count)) %>%
  group_by(Chromosome, Bin) %>%
  summarise(ClippingCount = sum(count), ClippingCountRaw = sum(raw_count), .groups = "drop") %>%
  ungroup()

  # Average read depth per bin
  read_depth_binned <- read_depth %>%
	group_by(chr, Bin) %>%
	summarize(ReadDepth = mean(depth), .groups = "drop") %>%
  ungroup()

  # change colname chr -> Chromosome
  setnames(read_depth_binned, "chr", "Chromosome")

  # Merge datasets fill 0
  combined <- full_join(clipping_counts, read_depth_binned,
					by = c("Chromosome", "Bin")) %>% 
					# fill with 0
					replace_na(list(ClippingCount = 0, ReadDepth = 0, ClippingCountRaw = 0)) %>%
          arrange(Bin)

  # Calculate clipping normalized by read depth
  combined$ClippingRate <- combined$ClippingCount / combined$ReadDepth

  # output as arrow dataset
  write_dataset(combined, output_dataset_dir, format = "arrow", partitioning = c("Chromosome"))
  print(paste0("Wrote ", ids$sample[i]))
  rm(read_depth)
  rm(read_depth_binned)
  gc(); gc()
}
  sort( sapply(ls(),function(x){object.size(get(x))})) 
}

[1] "Wrote L01_Anc"
[1] "Wrote L02_Anc"
[1] "Wrote L03_Anc"
[1] "Wrote L04_Anc"
[1] "Wrote L05_Anc"
[1] "Wrote L06_Anc"
[1] "Wrote L07_Anc"
[1] "Wrote L08_Anc"
[1] "Wrote L09_Anc"
[1] "Wrote L10_Anc"
[1] "Wrote L11_Anc"
[1] "Wrote L01-1_G08"
[1] "Wrote L01-2_G08"
[1] "Wrote L01-3_G08"
[1] "Wrote L01-4_G08"
[1] "Wrote L02-1_G08"
[1] "Wrote L02-2_G08"
[1] "Wrote L02-3_G08"
[1] "Wrote L02-4_G08"
[1] "Wrote L03-1_G08"
[1] "Wrote L03-2_G08"
[1] "Wrote L03-3_G08"
[1] "Wrote L03-4_G08"
[1] "Wrote L04-1_G08"
[1] "Wrote L04-2_G08"
[1] "Wrote L04-3_G08"
[1] "Wrote L04-4_G08"
[1] "Wrote L05-1_G08"
[1] "Wrote L05-2_G08"
[1] "Wrote L05-3_G08"
[1] "Wrote L05-4_G08"
[1] "Wrote L06-1_G08"
[1] "Wrote L06-2_G08"
[1] "Wrote L06-3_G08"
[1] "Wrote L06-4_G08"
[1] "Wrote L07-1_G08"
[1] "Wrote L07-2_G08"
[1] "Wrote L07-3_G08"
[1] "Wrote L07-4_G08"
[1] "Wrote L08-1_G08"
[1] "Wrote L08-2_G08"
[1] "Wrote L08-3_G08"
[1] "Wrote L08-4_G08"
[1] "Wrote L09-1_G08"
[1] "Wrote L09-2_G08"
[1] "Wrote L09-3_G08"
[1] "Wrote

i        window_size             sample       base_bam_dir 
                56                 56                120                136 
           fig_dir    is_position_csv       clip_end_dir        depth_ar_ds 
               136                136                152                152 
         depth_dir     is_pos_mds_csv output_dataset_dir output_summary_csv 
               152                152                152                152 
         clip_file                 ds    clipping_counts                ids 
               232                504              14528              58896 
     clipping_ends           combined 
            104088             171552

In [7]:
dcr <- open_dataset(output_dataset_dir, format = "arrow")
dcr %>% head %>% collect

Bin,ClippingCount,ClippingCountRaw,ReadDepth,ClippingRate,Chromosome
<dbl>,<int>,<int>,<dbl>,<dbl>,<chr>
0,30,30,30.13727,0.995445,L01-1_G08
1000,0,0,28.56700,0.000000,L01-1_G08
2000,0,0,28.71900,0.000000,L01-1_G08
3000,0,0,29.47400,0.000000,L01-1_G08
4000,0,0,32.78300,0.000000,L01-1_G08
5000,0,0,32.08700,0.000000,L01-1_G08


In [8]:
dcr %>% select(Chromosome) %>% distinct %>% collect %>% pull %>% sort

[1] "L01_Anc"           "L01-1_G08"         "L01-1_G20"        
  [4] "L01-1_G8_contig_1" "L01-1_G8_contig_3" "L01-2_G08"        
  [7] "L01-2_G20"         "L01-3_G08"         "L01-3_G20"        
 [10] "L01-4_G08"         "L01-4_G20"         "L02_Anc"          
 [13] "L02-1_G08"         "L02-1_G20"         "L02-1_G8_contig_1"
 [16] "L02-2_G08"         "L02-2_G20"         "L02-3_G08"        
 [19] "L02-3_G20"         "L02-4_G08"         "L02-4_G20"        
 [22] "L03_Anc"           "L03-1_G08"         "L03-1_G20"        
 [25] "L03-2_G08"         "L03-2_G20"         "L03-3_G08"        
 [28] "L03-3_G20"         "L03-4_G08"         "L03-4_G20"        
 [31] "L04_Anc"           "L04-1_G08"         "L04-1_G20"        
 [34] "L04-2_G08"         "L04-2_G20"         "L04-3_G08"        
 [37] "L04-3_G20"         "L04-4_G08"         "L04-4_G20"        
 [40] "L05_Anc"           "L05-1_G08"         "L05-1_G20"        
 [43] "L05-2_G08"         "L05-2_G20"         "L05-3_G08"        
 [46] "L05-3_G20"         "L05-4_G08"         "L05-4_G20"        
 [49] "L06_Anc"           "L06-1_G08"         "L06-1_G20"        
 [52] "L06-2_G08"         "L06-2_G20"         "L06-3_G08"        
 [55] "L06-3_G20"         "L06-4_G08"         "L06-4_G20"        
 [58] "L07_Anc"           "L07-1_G08"         "L07-1_G20"        
 [61] "L07-2_G08"         "L07-2_G20"         "L07-3_G08"        
 [64] "L07-3_G20"         "L07-4_G08"         "L07-4_G20"        
 [67] "L08_Anc"           "L08-1_G08"         "L08-1_G20"        
 [70] "L08-2_G08"         "L08-2_G20"         "L08-3_G08"        
 [73] "L08-3_G20"         "L08-4_G08"         "L08-4_G20"        
 [76] "L09_Anc"           "L09-1_G08"         "L09-1_G20"        
 [79] "L09-2_G08"         "L09-2_G20"         "L09-3_G08"        
 [82] "L09-3_G20"         "L09-4_G08"         "L09-4_G20"        
 [85] "L10_Anc"           "L10-1_G08"         "L10-1_G20"        
 [88] "L10-2_G08"         "L10-2_G20"         "L10-3_G08"        
 [91] "L10-3_G20"         "L10-4_G08"         "L10-4_G20"        
 [94] "L11_Anc"           "L11-1_G08"         "L11-1_G20"        
 [97] "L11-2_G08"         "L11-2_G20"         "L11-3_G08"        
[100] "L11-3_G20"         "L11-4_G08"         "L11-4_G20"        
[103] "YK_plasmid_3B2"

In [9]:
# import arrow dataset and create plots
dcr %>% filter(Chromosome == "L07-4_G08") %>% collect %>% tail

Bin,ClippingCount,ClippingCountRaw,ReadDepth,ClippingRate,Chromosome
<dbl>,<int>,<int>,<dbl>,<dbl>,<chr>
3989000,0,0,38.15500,0.0000000,L07-4_G08
3990000,0,0,39.74100,0.0000000,L07-4_G08
3991000,0,0,40.34300,0.0000000,L07-4_G08
3992000,0,0,37.85600,0.0000000,L07-4_G08
3993000,0,0,37.15300,0.0000000,L07-4_G08
3994000,35,35,36.08059,0.9700508,L07-4_G08


In [10]:
dcr %>% filter(Chromosome == "L01-4_G20", ClippingRate > 0.6 ) %>% collect %>% tail

Bin,ClippingCount,ClippingCountRaw,ReadDepth,ClippingRate,Chromosome
<dbl>,<int>,<int>,<dbl>,<dbl>,<chr>
0,31,31,30.16132,1.0278064,L01-4_G20
1420000,1,1,0.98700,1.0131712,L01-4_G20
1945000,11,11,14.83500,0.7414897,L01-4_G20
2082000,10,10,12.99800,0.7693491,L01-4_G20
4145000,31,31,29.96040,1.0346993,L01-4_G20


In [11]:
clip_file_1 <- file.path(clip_end_dir, paste0("filtered_clipped_ends_", "L01-1_G08", ".txt"))
clip_ends_1 <- fread(clip_file_1)
clip_ends_1 %>% head(2)

Chromosome,ReadName,Position,ClippingType,ClippingLength,ReadDir,End,Length,AlignedLength
<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<int>,<int>
L01-1_G08,1ab8fceb-b269-4baf-a0b8-fac869124be2,1,H,22695,+,Start,16269,16269
L01-1_G08,7056a7d4-7c6f-4cf0-af8b-da5cc3c033a8,1,H,1337,+,Start,1008,1008


## Get the positions of ISs

In [12]:
is_pos <- read_csv(is_position_csv) %>%
    separate(Line, into = c("ParentLine", "Subline"), sep = "-",remove = FALSE) %>%
    filter(!((Gen == 1) & (Subline > 1))) %>% # only use the Anc of the first subline
    mutate(Chromosome = case_when(
        Gen == 1 ~ paste0(ParentLine, "_Anc"),
        Gen == 2 ~ paste0(Line, "_G08"),
        Gen == 3 ~ paste0(Line, "_G20")
    ))
is_pos %>% head

Rows: 2384 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (2): IS_strand, Line
dbl (6): start, end, max_alignment_length, length, cluster_id, Gen

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


start,end,IS_strand,max_alignment_length,length,cluster_id,Line,ParentLine,Subline,Gen,Chromosome
<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>
476928,480020,reverse,3093,3093,0,L01-1,L01,1,1,L01_Anc
557167,560259,reverse,3093,3093,1,L01-1,L01,1,1,L01_Anc
1187831,1192502,forward,3093,4672,2,L01-1,L01,1,1,L01_Anc
1405123,1408215,forward,3093,3093,3,L01-1,L01,1,1,L01_Anc
1499230,1503892,reverse,3093,4663,4,L01-1,L01,1,1,L01_Anc
1986448,1991110,forward,3093,4663,5,L01-1,L01,1,1,L01_Anc


In [13]:
is_pos_mds <- read_csv(is_pos_mds_csv)
is_pos_mds %>% head

Rows: 5031 Columns: 12
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (5): flank, genome, position_status, sv, Line
dbl (5): pos_id, pos, cluster_id, insert_id, Gen
lgl (2): IS_strand, flankMatchDir

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


pos_id,IS_strand,flank,pos,genome,cluster_id,insert_id,flankMatchDir,position_status,sv,Line,Gen
<dbl>,<lgl>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<lgl>,<chr>,<chr>,<chr>,<dbl>
0,FALSE,f,1272610,Ref,5,4,FALSE,original,pristine,L01-1,1
0,FALSE,r,1275701,Ref,6,5,TRUE,original,pristine,L01-1,1
0,FALSE,r,476936,Query,2,1,TRUE,new,simple_insertion,L01-1,1
0,FALSE,f,476923,Query,2,1,FALSE,new,simple_insertion,L01-1,1
1,FALSE,r,554076,Query,4,2,TRUE,new,simple_insertion,L01-1,1
1,FALSE,f,554063,Query,4,2,FALSE,new,simple_insertion,L01-1,1


In [14]:
# Identify the IS that correspond to the adapter sequence pos_id == 0, genome == REF -> find the one with the same cluster_id
df <- is_pos_mds %>% mutate(LG = paste0(Line, "_", Gen)) 
LGs <- unique(df$LG)
outdf <- data.frame()
for (lg in LGs) {
    df_line <- filter(df, LG == lg)
    cluster_id_original <- df_line %>% filter(pos_id == 0, genome == "Ref") %>% pull(cluster_id) %>% unique
    outdf <- outdf %>% rbind(
        df_line %>% filter(cluster_id %in% cluster_id_original, genome == "Query")
    )
}
is_pos_mds_original_is <- outdf
is_pos_mds_original_is %>% head

pos_id,IS_strand,flank,pos,genome,cluster_id,insert_id,flankMatchDir,position_status,sv,Line,Gen,LG
<dbl>,<lgl>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<lgl>,<chr>,<chr>,<chr>,<dbl>,<chr>
3,FALSE,f,1275699,Query,6,5,FALSE,original,pristine,L01-1,1,L01-1_1
3,FALSE,r,1272611,Query,5,4,TRUE,original,pristine,L01-1,1,L01-1_1
5,FALSE,f,1275699,Query,8,7,FALSE,original,pristine,L01-1,2,L01-1_2
5,FALSE,r,1272611,Query,9,6,TRUE,original,pristine,L01-1,2,L01-1_2
7,FALSE,f,1275699,Query,28,8,FALSE,original,pristine,L01-1,3,L01-1_3
7,FALSE,r,1272611,Query,18,7,TRUE,original,pristine,L01-1,3,L01-1_3


In [15]:
is_pos_mds_original_is_clean <- is_pos_mds_original_is %>% select(cluster_id = pos_id, Line, Gen) %>% distinct 
is_pos_mds_original_is_clean %>% head

cluster_id,Line,Gen
<dbl>,<chr>,<dbl>
3,L01-1,1
5,L01-1,2
7,L01-1,3
3,L01-2,1
5,L01-2,2
5,L01-2,3


In [16]:
is_pos_mds_original_is_clean %>% filter(Line == "L03-2")

cluster_id,Line,Gen
<dbl>,<chr>,<dbl>
2,L03-2,1
7,L03-2,2
7,L03-2,3


In [17]:
# Manually check these regions that have multiple ISs, which have IS ends corresponding ot the lambda-red inserted copy, are indeed subject to SVs.
# All three are composite transposons
is_pos_mds_original_is_clean %>% group_by(Line, Gen) %>% mutate(n = n()) %>% filter(n > 1) %>% head

cluster_id,Line,Gen,n
<dbl>,<chr>,<dbl>,<int>
5,L02-1,3,2
23,L02-1,3,2
9,L04-2,3,2
10,L04-2,3,2
9,L04-3,2,2
12,L04-3,2,2


In [18]:
is_pos_mds_original_is_clean %>% filter(Line == "L01-1", Gen == 2) %>% head

cluster_id,Line,Gen
<dbl>,<chr>,<dbl>
5,L01-1,2


In [19]:
#start	end	IS_strand	max_alignment_length	length	cluster_id	Line	ParentLine	Subline	Gen	Chromosome
is_pos_mds_original_is_clean_range <- is_pos_mds_original_is_clean %>% 
    inner_join(
        is_pos %>% select(start, end, cluster_id, Line, Gen, Chromosome),
    )
is_pos_mds_original_is_clean_range %>% head(10)

Joining with `by = join_by(cluster_id, Line, Gen)`


cluster_id,Line,Gen,start,end,Chromosome
<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
3,L01-1,1,1405123,1408215,L01_Anc
5,L01-1,2,1406378,1409469,L01-1_G08
7,L01-1,3,1412099,1415191,L01-1_G20
5,L01-2,2,1372270,1375361,L01-2_G08
5,L01-2,3,1360211,1363303,L01-2_G20
4,L01-3,2,1391489,1394580,L01-3_G08
5,L01-3,3,1393493,1396585,L01-3_G20
3,L01-4,2,1405117,1408209,L01-4_G08
8,L01-4,3,1418450,1421542,L01-4_G20


## Create Figs
Identify all the ISs that correspond to the original copy, as reads corresponding to the region is ommited due to adapter filtering

In [20]:
anc_line_lst <- 
list(
	"R01" = "L01",
	"R02" = "L02",
	"R03" = "L03",
	"R04" = "L04",
	"R05" = "L05",
	"R06" = "L06",
	"r01" = "L07",
	"r02" = "L08",
	"r03" = "L09",
	"r04" = "L10",
	"r05" = "L11"
)

In [21]:
ids %>% pull(sample)

[1] "L01_Anc"   "L02_Anc"   "L03_Anc"   "L04_Anc"   "L05_Anc"   "L06_Anc"  
 [7] "L07_Anc"   "L08_Anc"   "L09_Anc"   "L10_Anc"   "L11_Anc"   "L01-1_G08"
[13] "L01-2_G08" "L01-3_G08" "L01-4_G08" "L02-1_G08" "L02-2_G08" "L02-3_G08"
[19] "L02-4_G08" "L03-1_G08" "L03-2_G08" "L03-3_G08" "L03-4_G08" "L04-1_G08"
[25] "L04-2_G08" "L04-3_G08" "L04-4_G08" "L05-1_G08" "L05-2_G08" "L05-3_G08"
[31] "L05-4_G08" "L06-1_G08" "L06-2_G08" "L06-3_G08" "L06-4_G08" "L07-1_G08"
[37] "L07-2_G08" "L07-3_G08" "L07-4_G08" "L08-1_G08" "L08-2_G08" "L08-3_G08"
[43] "L08-4_G08" "L09-1_G08" "L09-2_G08" "L09-3_G08" "L09-4_G08" "L10-1_G08"
[49] "L10-2_G08" "L10-3_G08" "L10-4_G08" "L11-1_G08" "L11-2_G08" "L11-3_G08"
[55] "L11-4_G08" "L01-1_G20" "L01-2_G20" "L01-3_G20" "L01-4_G20" "L02-1_G20"
[61] "L02-2_G20" "L02-3_G20" "L02-4_G20" "L03-1_G20" "L03-2_G20" "L03-3_G20"
[67] "L03-4_G20" "L04-1_G20" "L04-2_G20" "L04-3_G20" "L04-4_G20" "L05-1_G20"
[73] "L05-2_G20" "L05-3_G20" "L05-4_G20" "L06-1_G20" "L06-2_G20" "L06-3_G20"
[79] "L06-4_G20" "L07-1_G20" "L07-2_G20" "L07-3_G20" "L07-4_G20" "L08-1_G20"
[85] "L08-2_G20" "L08-3_G20" "L08-4_G20" "L09-1_G20" "L09-2_G20" "L09-3_G20"
[91] "L09-4_G20" "L10-1_G20" "L10-2_G20" "L10-3_G20" "L10-4_G20" "L11-1_G20"
[97] "L11-2_G20" "L11-3_G20" "L11-4_G20"

In [22]:
# create figures with anc
clipping_ratio_threshold_min <- 0.6
clipping_ratio_threshold_max <- 2.1
#Lines = ids %>% pull(Line) %>% unique
Ancs <- ids %>% pull(Anc) %>% unique

png_dir <- file.path(fig_dir, 'clip_rate', 'png')
dir.create(png_dir, recursive = TRUE)
pdf_dir <- file.path(fig_dir, 'clip_rate', 'pdf')
dir.create(pdf_dir, recursive = TRUE)

#for (line in Lines) {
for (anc in Ancs) {
	if (anc != "r02") {
		next
	}
	samples <- ids %>% filter(Anc == anc) %>% pull(sample)
	line <- anc_line_lst[[anc]]
	# valid rectangle xmin,xmax <= min max values, ymin,ymax clipping thresholds for each sample
	valid_rect <- dcr %>%
		filter(Chromosome %in% samples) %>%
		collect %>%
		group_by(Chromosome) %>%
		summarise(xmin = min(Bin), xmax = max(Bin), ymin = clipping_ratio_threshold_min, ymax = clipping_ratio_threshold_max, .groups = "drop")

	original_is_rect <- is_pos_mds_original_is_clean_range %>%
		filter(Chromosome %in% samples) %>%
		mutate(xmin = start, xmax = end, ymin = 0.0, ymax = clipping_ratio_threshold_max+0.2) %>%
		mutate(x = (start+end)/2)

	p <- dcr %>%
		filter(Chromosome %in% samples) %>%
		collect %>%
		mutate(ClippingRate = ifelse(ClippingRate > clipping_ratio_threshold_max, clipping_ratio_threshold_max, ClippingRate)) %>%
		ggplot(aes(x = Bin/1e6, y = ClippingRate)) +
		geom_point(size = 2, shape = 1) +
		geom_point(data = is_pos %>% mutate(Bin = (start+end)/2, ClippingRate = clipping_ratio_threshold_max+0.1) %>% filter(Chromosome %in% samples), shape = 3, size = 2, color = "blue") +
		theme_half_open(10) +
		labs(x = "Position (1kbp window, Mbp)", y = "Clipping ends / Mean read depth") +
		facet_wrap(~Chromosome) +
		geom_rect(data = valid_rect, aes(xmin = xmin/1e6, xmax = xmax/1e6, ymin = ymin, ymax = ymax, x = NULL, y = NULL), fill = "blue", alpha = 0.1) +
		#geom_rect(data = original_is_rect, aes(xmin = xmin/1e6, xmax = xmax/1e6, ymin = ymin, ymax = ymax, x = NULL, y = NULL), fill = "grey", alpha = 0.1)
		geom_segment(data = original_is_rect, aes(x = x/1e6, xend = x/1e6, y = ymin, yend = ymax), color = "grey", size = 0.5)
	ggsave(file.path(png_dir, paste0(line, "_clipping_rate.png")), width = 10, height = 10, plot = p)
	ggsave(file.path(pdf_dir, paste0(line, "_clipping_rate.pdf")), width = 10, height = 10, plot = p)
	print(samples)
}

Warning message in dir.create(png_dir, recursive = TRUE):
“'../exp/fig/clipping/cutadapt/clip_rate/png' already exists”
Warning message in dir.create(pdf_dir, recursive = TRUE):
“'../exp/fig/clipping/cutadapt/clip_rate/pdf' already exists”
Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.”


Warning message:
“Removed 7 rows containing missing values (`geom_point()`).”
Warning message:
“Removed 7 rows containing missing values (`geom_point()`).”


[1] "L08_Anc"   "L08-1_G08" "L08-2_G08" "L08-3_G08" "L08-4_G08" "L08-1_G20"
[7] "L08-2_G20" "L08-3_G20" "L08-4_G20"


## Stats

In [23]:
cluster_bins <- function(df, is_pos_mds_original_is, bin_size = 100000,
    overlap_buffer = 10000) {
    # initialize
    df <- df %>% arrange(Chromosome, Bin)
    df$End <- 0
    df$Original <- 0

    # Split the data by Chromosome and Process Each Separately
    clustered_data <- df %>%
        group_by(Chromosome) %>%
        mutate(Bin_ = Bin) %>%
        mutate(Bin = case_when(
            Bin_ == min(Bin_) ~ Bin_ - bin_size * 2,
            Bin_ == max(Bin_) ~ Bin_ + bin_size * 2,
            TRUE ~ Bin_
        )) %>%
        mutate(
            # Identify Clusters
            Cluster = cumsum(c(1, diff(Bin) > bin_size)),
            
            # Mark First and Last Bin in Each Chromosome
            End = ifelse(Bin == min(Bin) | Bin == max(Bin), 1, 0)
        ) %>% ungroup() %>% mutate(Bin = Bin_) %>% select(-Bin_)

    # Check for Overlaps with IS Regions (by Chromosome)
    for (chrom in unique(df$Chromosome)) {
        print(chrom)
        # Filter current chromosome bins and IS regions
        chrom_bins <- clustered_data %>% filter(Chromosome == chrom)
        chrom_is <- is_pos_mds_original_is %>% filter(Chromosome == chrom)
        #print(chrom_bins)
        #print(chrom_is)
        
        # Convert to IRanges for overlap checking
        bin_ranges <- IRanges(start = chrom_bins$Bin, end = chrom_bins$Bin)
        is_ranges <- IRanges(start = chrom_is$start - overlap_buffer, end = chrom_is$end + overlap_buffer)
        
        # Find overlaps
        overlaps <- findOverlaps(bin_ranges, is_ranges)
        print(overlaps)
        
        # Mark Original as 1 for bins within IS regions
        if (length(overlaps) > 0) {
            # Use queryHits to access the bins being overlapped
            overlap_indices <- queryHits(overlaps)
            clustered_data$Original[clustered_data$Chromosome == chrom][overlap_indices] <- 1
        }
    }
    
    return(clustered_data)
}

In [24]:
clustered_clipping_rates <- cluster_bins(dcr %>% filter(ClippingRate > 0.6) %>% collect, is_pos_mds_original_is_clean_range,
    bin_size = 100000)
clustered_clipping_rates %>% head

[1] "L01-1_G08"
Hits object with 0 hits and 0 metadata columns:
   queryHits subjectHits
   <integer>   <integer>
  -------
  queryLength: 2 / subjectLength: 1
[1] "L01-1_G20"
Hits object with 1 hit and 0 metadata columns:
      queryHits subjectHits
      <integer>   <integer>
  [1]         2           1
  -------
  queryLength: 3 / subjectLength: 1
[1] "L01-1_G8_contig_1"
Hits object with 0 hits and 0 metadata columns:
   queryHits subjectHits
   <integer>   <integer>
  -------
  queryLength: 2 / subjectLength: 0
[1] "L01-1_G8_contig_3"
Hits object with 0 hits and 0 metadata columns:
   queryHits subjectHits
   <integer>   <integer>
  -------
  queryLength: 7 / subjectLength: 0
[1] "L01-2_G08"
Hits object with 2 hits and 0 metadata columns:
      queryHits subjectHits
      <integer>   <integer>
  [1]         2           1
  [2]         3           1
  -------
  queryLength: 4 / subjectLength: 1
[1] "L01-2_G20"
Hits object with 1 hit and 0 metadata columns:
      queryHits subjectHit

Bin,ClippingCount,ClippingCountRaw,ReadDepth,ClippingRate,Chromosome,End,Original,Cluster
<dbl>,<int>,<int>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>
0,30,30,30.13727,0.995445,L01-1_G08,1,0,1
4051000,30,30,29.48095,1.017606,L01-1_G08,1,0,2
0,28,28,27.51102,1.017774,L01-1_G20,1,0,1
1415000,7,11,4.14500,1.688782,L01-1_G20,0,1,2
3923000,28,28,27.22571,1.028440,L01-1_G20,1,0,3
9000,1,1,0.00000,Inf,L01-1_G8_contig_1,1,0,1


In [25]:
# summary
sprintf("Number of bins: %d", nrow(clustered_clipping_rates))
c_df <- clustered_clipping_rates %>% group_by(Cluster, Chromosome) %>% summarize(Remove = End + Original, End = sum(End), Original = sum(Original), .groups = "drop") %>%
    left_join(is_pos %>% select(Line:Chromosome) %>% distinct) %>% ungroup
sprintf("Number of clusters: %d", nrow(c_df))
sprintf("Number of clusters not End: %d", nrow(filter(c_df, End == 0)))
sprintf("Number of clusters not Original IS nor End: %d", nrow(filter(c_df, Remove == 0)))
sprintf("Nor Anc.: %d", nrow(filter(c_df, Gen != 1 , Remove == 0)))

[1] "Number of bins: 283"

Warning message:
“Returning more (or less) than 1 row per `summarise()` group was deprecated in
dplyr 1.1.0.
ℹ Please use `reframe()` instead.
ℹ When switching from `summarise()` to `reframe()`, remember that `reframe()`
  always returns an ungrouped data frame and adjust accordingly.”
Joining with `by = join_by(Chromosome)`


[1] "Number of clusters: 283"

[1] "Number of clusters not End: 77"

[1] "Number of clusters not Original IS nor End: 35"

[1] "Nor Anc.: 5"

In [30]:
# show them
c_df %>% filter(Remove == 0, Gen != 1) %>% select(Cluster, Chromosome, Line, Gen) 

Cluster,Chromosome,Line,Gen
<dbl>,<chr>,<chr>,<dbl>
2,L05-2_G20,L05-2,3
2,L05-4_G08,L05-4,2
2,L06-1_G08,L06-1,2
3,L01-4_G20,L01-4,3
4,L01-4_G20,L01-4,3


In [26]:
write_csv(clustered_clipping_rates, output_summary_csv)

In [27]:
clustered_clipping_rates <- read_csv(output_summary_csv)
clustered_clipping_rates %>% head

Rows: 283 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): Chromosome
dbl (8): Bin, ClippingCount, ClippingCountRaw, ReadDepth, ClippingRate, End,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Bin,ClippingCount,ClippingCountRaw,ReadDepth,ClippingRate,Chromosome,End,Original,Cluster
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>
0,30,30,30.13727,0.995445,L01-1_G08,1,0,1
4051000,30,30,29.48095,1.017606,L01-1_G08,1,0,2
0,28,28,27.51102,1.017774,L01-1_G20,1,0,1
1415000,7,11,4.14500,1.688782,L01-1_G20,0,1,2
3923000,28,28,27.22571,1.028440,L01-1_G20,1,0,3
9000,1,1,0.00000,Inf,L01-1_G8_contig_1,1,0,1


In [28]:
clustered_clipping_rates %>% filter(Chromosome == "L03-1_G20") %>% collect 

Bin,ClippingCount,ClippingCountRaw,ReadDepth,ClippingRate,Chromosome,End,Original,Cluster
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>
0,40,40,39.27856,1.018367,L03-1_G20,1,0,1
1267000,14,14,2.28100,6.137659,L03-1_G20,0,1,2
3918000,40,40,39.28794,1.018124,L03-1_G20,1,0,3
